# Silver: home services

**Audience:** data engineers validating medallion architecture and AIDP lineage.

**Prerequisites:** the canonical lab assets, shared compute and five job parameters.

**Learning goals:** trace governed transformations, verify isolation, and inspect deterministic results.


In [ ]:
import re
from functools import reduce
from pyspark.sql import Window, functions as F

# oidlUtils is injected by AIDP Workbench; no import is required.
def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != "telco_lineage":
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

layer_prefixes = {"landing": "01_landing", "bronze": "02_bronze", "silver": "03_silver", "gold": "04_gold"}

def table(layer, logical_name):
    return f"aidp_lab.oci_{layer}.{participant_key}_{lab_id}_{logical_name}"

def location(layer, logical_name):
    return f"oci://{bucket_name}@{objectstorage_namespace}/{layer_prefixes[layer]}/users/{participant_key}/{lab_id}/{logical_name}/"

def write_delta(frame, layer, logical_name, _ddl):
    target = table(layer, logical_name)
    (frame.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target))
    actual = spark.table(target).count()
    assert actual == frame.count(), f"Delta count mismatch for {logical_name}"
    print(f"Delta {layer}.{logical_name}: {actual} rows")


## Transformation

Run this cell once. It is idempotent and checks its row-level contract.


In [ ]:
customers = spark.table(table("silver", "customer_master")).select("customer_id").withColumn("_customer_ok", F.lit(True))
products = (spark.table(table("silver", "product_catalog")).filter(F.col("service_type") == "HOME")
    .select("product_id", "monthly_fee").withColumn("_product_ok", F.lit(True)))
services = spark.table(table("bronze", "home_services"))
checked_services = services.join(customers, "customer_id", "left").join(products, "product_id", "left")
service_reason = (F.when(F.col("_customer_ok").isNull(), F.lit("orphan_customer"))
    .when(F.col("_product_ok").isNull(), F.lit("invalid_product")))
checked_services = checked_services.withColumn("_reason", service_reason)
service_issues = (checked_services.filter(F.col("_reason").isNotNull())
    .select(F.lit(participant_key).alias("participant_key"), F.lit("home_services").alias("dataset"),
        "source_row_id", F.col("service_id").alias("record_key"), F.col("_reason").alias("reason_code"),
        F.current_timestamp().alias("quarantined_at")))
valid_services = checked_services.filter(F.col("_reason").isNull())

installations = spark.table(table("bronze", "home_installations"))
installation_window = Window.partitionBy("installation_id").orderBy(F.col("updated_at").desc(), F.col("source_row_id").desc())
checked_installations = (installations.withColumn("_rank", F.row_number().over(installation_window))
    .join(valid_services.select("service_id").withColumn("_service_ok", F.lit(True)), "service_id", "left")
    .join(spark.table(table("silver", "customer_addresses")).select("address_id").withColumn("_address_ok", F.lit(True)), "address_id", "left"))
installation_reason = (F.when(F.col("_rank") > 1, F.lit("duplicate_installation"))
    .when(F.col("_service_ok").isNull(), F.lit("orphan_service"))
    .when(F.col("_address_ok").isNull(), F.lit("orphan_address")))
checked_installations = checked_installations.withColumn("_reason", installation_reason)
installation_issues = (checked_installations.filter(F.col("_reason").isNotNull())
    .select(F.lit(participant_key).alias("participant_key"), F.lit("home_installations").alias("dataset"),
        "source_row_id", F.col("installation_id").alias("record_key"), F.col("_reason").alias("reason_code"),
        F.current_timestamp().alias("quarantined_at")))
valid_installations = checked_installations.filter(F.col("_reason").isNull()).select("service_id", "address_id", "technology")
home_service = (valid_services.join(valid_installations, "service_id", "left")
    .select("participant_key", "service_id", F.col("service_number"), F.lit("HOME").alias("service_type"),
        "customer_id", "product_id", F.lower("status").alias("status"),
        F.col("monthly_fee").cast("decimal(14,2)").alias("monthly_value"), "address_id", "technology"))
write_delta(home_service, "silver", "home_service", "participant_key STRING, service_id STRING, service_number STRING, service_type STRING, customer_id STRING, product_id STRING, status STRING, monthly_value DECIMAL(14,2), address_id STRING, technology STRING")
service_issues.unionByName(installation_issues).write.format("delta").mode("overwrite").save(location("silver", "_quality/home"))
assert home_service.count() == 218


## Exercise and common pitfall

**Exercise:** follow one customer or service identifier into the next task and explain every derived column.

**Answer scaffold:** identify the source table, join key, transformation and target column.

**Pitfall:** never replace the job parameters with participant-specific literals; doing so breaks canonical hashes and isolation.

**Extension:** inspect the resulting entity and column lineage in Master Catalog.
